# MimicGuard v2 / 9: R15 label-free tuning of the detectors, R16 wider slack grid and a tuned oracle

Simulation-only. R14 showed that tuning removes the ranking flips, but the tuning used attack data. Here each detector family is also tuned WITHOUT attack data, by a synthetic anomaly on normal check streams (two random features shifted by 0.4 and 0.75 training standard deviations), and the CUSUM slack grid goes down to 0. Policies compared on the same fresh test seeds: labelled global, label-free (global, per cell, per seed and cell), an uninformed random choice and the best configuration in hindsight. The oracle is given the same slack grid.

**Kaggle settings:** Accelerator = **None (CPU)**; Internet = **Off** (nothing is downloaded; no token or secret is needed);
no dataset needs to be added. Run with *Save Version -> Save & Run All (Commit)* for the long runs so the output is kept.
Set `MODE` in the config cell: `"smoke"` = a couple of minutes, to check that everything works; `"medium"` and `"full"` = the real runs.
Runs are resumable: if the session stops, run it again and finished units are skipped (the output folder `results_r15` must stay the same).

Estimated run time (not measured on Kaggle): about 1.5 to 2 hours on 4 CPUs (40 units, R14 took 48 min for the same units and this run adds synthetic streams and more slack values); MODE `smoke` first (about 3 to 6 min). Resumable.

In [ ]:
# ---- 1. Setup: unpack the embedded code, check the environment ----
import base64, io, os, sys, zipfile, subprocess, platform, warnings
warnings.filterwarnings("ignore")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("mimicguard_run")
os.makedirs(WORK, exist_ok=True)
REPO = os.path.join(WORK, "mimicguard")
_B64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_B64))) as z:
    z.extractall(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.chdir(REPO)
try:
    import yaml
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
import numpy, scipy, sklearn
print("python", platform.python_version(), "| numpy", numpy.__version__, "| sklearn", sklearn.__version__, "| CPUs", os.cpu_count())
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
except Exception as e:
    print("torch missing:", e)

In [ ]:
# ---- 2. Unit tests ----
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1800:], r.stderr[-800:])
assert r.returncode == 0, "unit tests failed: stop and send the output above"

In [ ]:
# ---- 3. Config ----
from mimicguard.config import load_config
from mimicguard.v2 import experiments as ex, r14
import pandas as pd, numpy as np
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 400)
BASE = load_config(os.path.join(REPO, "experiments/configs/base.yaml"))
MODE = "full"                    # "smoke" | "full"
N_JOBS = os.cpu_count() or 1
OUT = os.path.join(WORK, "results_r15"); os.makedirs(OUT, exist_ok=True)
TINY = dict(n_train=12, n_val=8, n_cal=20, n_chk=20, n_test=10, n_otrain=12, H=40, n_test_win=80, onset=40)
KS = r14.KS_EXT
if MODE == "smoke":
    CELLS = [("rayleigh", 20), ("none", 20)]; TUNE_SEEDS, TEST_SEEDS, SIZES = [100], [200], dict(TINY, n_chk=40)
else:
    CELLS = [("none", 20), ("rician", 20), ("rayleigh", 20), ("rayleigh", 15)]
    TUNE_SEEDS, TEST_SEEDS = [100, 101, 102, 103, 104], [200, 201, 202, 203, 204]   # same seeds as R14, so the two runs are comparable
    SIZES = dict(n_train=300, n_val=40, n_cal=500, n_chk=500, n_test=100, n_otrain=100, H=100, n_test_win=200, onset=100)
print(MODE, CELLS, "tune", TUNE_SEEDS, "test", TEST_SEEDS, "slack grid", KS)

In [ ]:
# ---- 4. Run: tuning seeds first (labelled selection), then test seeds; synthetic-anomaly columns for the label-free rule ----
tune = r14.run_r14(BASE, CELLS, TUNE_SEEDS, OUT, "tune", SIZES, ks=KS, n_jobs=N_JOBS, synthetic=True, tag="r15")
tune.to_csv(os.path.join(OUT, "r15_tune_rows.csv"), index=False)
test = r14.run_r14(BASE, CELLS, TEST_SEEDS, OUT, "test", SIZES, ks=KS, n_jobs=N_JOBS, synthetic=True, tag="r15")
test.to_csv(os.path.join(OUT, "r15_test_rows.csv"), index=False)
print(len(tune), len(test), "rows")

In [ ]:
# ---- 5. Analysis ----
assert set(tune.seed) & set(test.seed) == set(), "tuning and test seeds must be disjoint"
real_tune, real_test = tune[tune.family != "oracle_tuned"], test[test.family != "oracle_tuned"]
tab = r14.selection_table(real_tune, real_test)
tab.to_csv(os.path.join(OUT, "r15_selection_table.csv"), index=False)
summ = r14.ranking_summary(tab[tab.policy.isin(["global", "lf_global", "lf_cell", "lf_unit", "random", "hindsight"])])
summ.to_csv(os.path.join(OUT, "r15_ranking_summary.csv"), index=False)
chosen = pd.concat([r14.select(real_tune, "global").assign(policy="global")] +
                   [r14.select_label_free(real_test, pol).assign(policy=pol) for pol in ("lf_global", "lf_cell")])
chosen.to_csv(os.path.join(OUT, "r15_selected_configs.csv"), index=False)
for p in (0.05, 0.01):
    print(f"=== test Pd by selection policy (p = {p}) ===")
    print(tab[tab.p == p].pivot_table(index=["family", "policy"], columns=["fading", "snr_db"], values="pd").round(3))
    print(f"=== mean Pd over families and cells (p = {p}) ===")
    print(tab[tab.p == p].groupby("policy").pd.mean().round(3).sort_values())
    print(f"=== realised false alarm on check streams (p = {p}); target {p} ===")
    print(tab[tab.p == p].groupby("policy").fa_chk.agg(["mean", "min", "max"]).round(3))
    print(f"=== slack chosen (p = {p}) ===")
    print(chosen[chosen.p == p].groupby(["policy", "wrap"]).size().unstack(fill_value=0))
print("=== ranking summary ===")
print(summ.to_string(index=False))
# R16: oracle with the same slack grid, selected with attack data on the tuning seeds
ot = tune[tune.family == "oracle_tuned"].copy(); ot["sel"] = ot.pd + 0.01 * ot.auc
for p in (0.05, 0.01):
    best_wrap = ot[ot.p == p].groupby("wrap").sel.mean().idxmax()
    o = test[(test.family == "oracle_tuned") & (test.p == p) & (test.wrap == best_wrap)].groupby(["fading", "snr_db"]).pd.mean()
    print(f"=== oracle, slack tuned on tuning seeds (p = {p}): {best_wrap} ===")
    print(o.round(3))

In [ ]:
# ---- last cell: package outputs ----
import shutil
zp = shutil.make_archive(os.path.join(WORK, "mimicguard_v2_r15_results"), "zip", OUT)
print("results archive:", zp, f"({os.path.getsize(zp)/1e6:.1f} MB)")
print("Download it from the Output tab (after Save Version -> Save & Run All), then send it back.")